# 10 — Pipeline integrado, retomável e auditável

Este notebook é uma interface para a mesma biblioteca usada no servidor. Ele não reimplementa codecs, métricas ou seleção de dados. As células pesadas permanecem desativadas por padrão.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'freeze-spec-v1.json').is_file():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / 'src'))
from codeco.codeco_path import CodecoPath
from codeco.experiment_config import experiment_config
from codeco.protocol import PROTOCOL, adapter_audit

print('Repositório:', ROOT)
print('Protocolo:', PROTOCOL.as_dict())
print('Adaptadores:', adapter_audit())

## Fontes de verdade

A configuração abaixo alimenta os scripts antigos e o runner otimizado. Os manifests CSV e o lock continuam sendo a definição autoritativa dos corpora.

In [ ]:
config = experiment_config()
display(config)
subprocess.run([sys.executable, str(ROOT/'scripts/verify_frozen_install.py'), '--stage', 'metadata'], check=True)

## Backend disponível

CUDA acelera as etapas neurais, mas não é obrigatória. Em sua ausência, os mesmos estágios usam CPU e mantêm checkpoints compatíveis.

In [ ]:
try:
    import torch
    backend = 'cuda' if torch.cuda.is_available() else ('mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu')
except ImportError:
    backend = 'dependências neurais ainda não instaladas'
print('Backend:', backend)

## Execução

A execução completa valida o freeze, registra commit, hashes, ambiente e hardware; depois usa cache PCM, checkpoints e gravação atômica. Ative somente após os notebooks 01–04.

In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    from codeco.main import Codeco
    run = (Codeco().environment().datasets().flops().prepare_baseline()
           .ffmpeg().encodec().embeddings().classify().analyze().graphs())
    display(run.context)
else:
    print('Execução desativada. Defina RUN_FULL_PIPELINE=True conscientemente.')

## Artefatos esperados

- `results/run-metadata.json`: commit, hashes, versões e hardware;
- `results/tables/`: métricas, classificação, erros pareados e Pareto;
- `data/checkpoints/`: retomada;
- `data/cache/`: PCM, reconstruções e embeddings;
- `results/logs/`: comandos, tempos, falhas e retries.